# YOLO OBB 학습 (주피터용, 단일 모델)

컴퓨터 3대에서 각각 다른 모델을 돌리기 위한 노트북입니다.

**사용법**: 1번 셀에서 `MODEL` 변수만 바꿔서 각 컴퓨터에서 실행하세요.
- 컴퓨터 1: `MODEL = "yolov8n-obb.pt"`
- 컴퓨터 2: `MODEL = "yolo11n-obb.pt"`
- 컴퓨터 3: `MODEL = "yolo26n-obb.pt"`

## 1. 설정 (여기만 수정하세요)

In [2]:
# ⬇️ 이 컴퓨터에서 학습할 모델 (컴퓨터마다 다르게)
MODEL = "yolo11n-obb.pt"

# ⬇️ 데이터셋 경로
DATASET_ROOT = "/data/yolo_obb_compressed"

# ⬇️ 학습 하이퍼파라미터
EPOCHS    = 50
BATCH     = 16        # GPU 메모리에 맞게 조절
IMGSZ     = 960
DEVICE    = 0         # GPU 번호. CPU면 "cpu"
WORKERS   = 8
PATIENCE  = 20        # early stopping
LR0       = 0.01
SEED      = 42
CACHE     = False     # True = RAM 캐시 (메모리 넉넉하면 빠름)
AMP       = True      # mixed precision

# ⬇️ 결과 저장 위치
PROJECT_DIR = "/data/runs"

# ⬇️ 클래스 이름 30개 (실제 클래스명으로 교체하세요!)
CLASS_NAMES = [
    "BenchWithBack_Normal",       # 0
    "BenchWithBack_Damaged",      # 1
    "BenchWithoutBack_Normal",    # 2
    "BenchWithoutBack_Damaged",   # 3
    "ProtectionFence_Normal",     # 4
    "ProtectionFence_Damaged",    # 5
    "Bollard_Normal",             # 6
    "Bollard_Damaged",            # 7
    "JaywalkPrevention_Normal",   # 8
    "JaywalkPrevention_Damaged",  # 9
    "TreeCover_Normal",           # 10
    "TreeCover_Damaged",          # 11
    "Manhole_Normal",             # 12
    "Manhole_Damaged",            # 13
    "Trench_Normal",              # 14
    "Trench_Damaged",             # 15
    "CurbStone_Normal",           # 16
    "CurbStone_Damaged",          # 17
    "SidewalkBlock_Normal",       # 18
    "SidewalkBlock_Damaged",      # 19
    "BrailleBlock_Normal",        # 20
    "BrailleBlock_Damaged",       # 21
]
assert len(CLASS_NAMES) == 22, f"클래스 개수 오류: {len(CLASS_NAMES)}"
print(f"🎯 학습 모델: {MODEL}")
print(f"📁 데이터셋: {DATASET_ROOT}")
print(f"🏷️  클래스 수: {len(CLASS_NAMES)}")

🎯 학습 모델: yolo11n-obb.pt
📁 데이터셋: /data/yolo_obb_compressed
🏷️  클래스 수: 22


## 2. dataset.yaml 생성

In [3]:
from pathlib import Path

dataset_root = Path(DATASET_ROOT)
yaml_path = dataset_root / "dataset.yaml"

# yaml 내용 생성
yaml_content = f"path: {dataset_root}\ntrain: train/images\nval: val/images\n\nnames:\n"
for i, n in enumerate(CLASS_NAMES):
    yaml_content += f"  {i}: {n}\n"

yaml_path.write_text(yaml_content, encoding="utf-8")
print(f"✅ yaml 생성: {yaml_path}\n")
print(yaml_content)

✅ yaml 생성: /data/yolo_obb_compressed/dataset.yaml

path: /data/yolo_obb_compressed
train: train/images
val: val/images

names:
  0: BenchWithBack_Normal
  1: BenchWithBack_Damaged
  2: BenchWithoutBack_Normal
  3: BenchWithoutBack_Damaged
  4: ProtectionFence_Normal
  5: ProtectionFence_Damaged
  6: Bollard_Normal
  7: Bollard_Damaged
  8: JaywalkPrevention_Normal
  9: JaywalkPrevention_Damaged
  10: TreeCover_Normal
  11: TreeCover_Damaged
  12: Manhole_Normal
  13: Manhole_Damaged
  14: Trench_Normal
  15: Trench_Damaged
  16: CurbStone_Normal
  17: CurbStone_Damaged
  18: SidewalkBlock_Normal
  19: SidewalkBlock_Damaged
  20: BrailleBlock_Normal
  21: BrailleBlock_Damaged



## 3. 환경 확인

In [4]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"ultralytics: {ultralytics.__version__}")
print(f"torch:       {torch.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available() and isinstance(DEVICE, int):
    props = torch.cuda.get_device_properties(DEVICE)
    print(f"GPU:         {props.name}")
    print(f"VRAM:        {props.total_memory / 1e9:.1f} GB")

# 데이터셋 무결성 체크
print("\n[데이터셋 확인]")
for split in ["train", "val"]:
    img_dir = dataset_root / split / "images"
    lbl_dir = dataset_root / split / "labels"
    img_cnt = len([p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png"}]) if img_dir.exists() else 0
    lbl_cnt = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"  [{split}] images: {img_cnt:,} / labels: {lbl_cnt:,}")

ultralytics: 8.4.40
torch:       2.11.0+cu130
CUDA:        True
GPU:         NVIDIA GeForce RTX 2080
VRAM:        8.2 GB

[데이터셋 확인]
  [train] images: 35,640 / labels: 35,640
  [val] images: 3,919 / labels: 3,919


## 4. 학습 실행
**⚠️ 이 셀은 오래 걸립니다** (모델 크기와 데이터에 따라 수시간 ~ 하루)

이미 학습된 `best.pt` 가 있으면 자동으로 건너뛰어요.

In [5]:
import time
import gc

run_name = MODEL.replace(".pt", "")
run_dir = Path(PROJECT_DIR) / run_name
best_weights = run_dir / "weights" / "best.pt"

print(f"📂 실행 디렉토리: {run_dir}")

if best_weights.exists():
    print(f"⏭️  이미 학습 완료: {best_weights}")
    print(f"    재학습하려면 run_dir 를 삭제하고 다시 실행하세요.")
else:
    model = YOLO(MODEL)

    t0 = time.time()
    model.train(
        data     = str(yaml_path),
        epochs   = EPOCHS,
        imgsz    = IMGSZ,
        batch    = BATCH,
        device   = DEVICE,
        workers  = WORKERS,
        patience = PATIENCE,
        lr0      = LR0,
        seed     = SEED,
        cache    = CACHE,
        amp      = AMP,
        plots    = True,
        project  = PROJECT_DIR,
        name     = run_name,
        exist_ok = True,
    )

    train_min = (time.time() - t0) / 60
    print(f"\n✅ 학습 완료: {train_min:.1f}분")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

📂 실행 디렉토리: /data/runs/yolo11n-obb
Ultralytics 8.4.40 🚀 Python-3.10.20 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2080, 7786MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/data/yolo_obb_compressed/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n-obb, nbs=64, nms=False, opset=None, o

## 5. 검증 + 상세 평가

In [ ]:
assert best_weights.exists(), f"best.pt 없음: {best_weights}"

print(f"📊 검증 시작: {best_weights}\n")
val_model = YOLO(str(best_weights))
metrics = val_model.val(
    data      = str(yaml_path),
    imgsz     = IMGSZ,
    device    = DEVICE,
    plots     = True,
    save_json = False,
)

print("\n[전체 평균]")
print(f"  Precision (mean): {metrics.box.mp:.4f}")
print(f"  Recall    (mean): {metrics.box.mr:.4f}")
print(f"  mAP@50         : {metrics.box.map50:.4f}")
print(f"  mAP@75         : {metrics.box.map75:.4f}")
print(f"  mAP@50-95      : {metrics.box.map:.4f}")

speed_total = sum(metrics.speed.values())
print(f"\n[속도]")
for k, v in metrics.speed.items():
    print(f"  {k:12s}: {v:6.2f} ms/image")
print(f"  {'total':12s}: {speed_total:6.2f} ms/image")

📊 검증 시작: /data/runs/yolo11n-obb/weights/best.pt

Ultralytics 8.4.40 🚀 Python-3.10.20 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2080, 7786MiB)
YOLO11n-obb summary (fused): 110 layers, 2,658,013 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3735.4±702.4 MB/s, size: 235.9 KB)
val: Scanning /data/yolo_obb_compressed/val/labels.cache... 3919 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3919/3919 632.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 245/245 5.2it/s 47.2s<0.2s
                   all       3919       4179      0.884      0.872      0.921      0.807
  BenchWithBack_Normal        300        304      0.904       0.99      0.985      0.969
 BenchWithBack_Damaged         60         60      0.934      0.712      0.843      0.802
BenchWithoutBack_Normal        300        305      0.939      0.987      0.989      0.955
BenchWithoutBack_Damaged         60         60 

: 

## 6. 클래스별 상세 지표

In [ ]:
import pandas as pd

p = metrics.box.p           # precision per class
r = metrics.box.r           # recall per class
ap50 = metrics.box.ap50     # AP@50 per class
ap = metrics.box.ap         # AP@50:95 per class
if ap.ndim == 2:
    ap = ap.mean(axis=1)

rows = []
for i, name in enumerate(CLASS_NAMES):
    if i < len(p):
        rows.append({
            "class_id":   i,
            "class_name": name,
            "precision":  float(p[i]),
            "recall":     float(r[i]),
            "mAP50":      float(ap50[i]),
            "mAP50-95":   float(ap[i]),
        })
    else:
        rows.append({
            "class_id":   i,
            "class_name": name,
            "precision":  None, "recall": None,
            "mAP50":      None, "mAP50-95": None,
        })

df = pd.DataFrame(rows)
df_display = df.round(4)

# 색칠된 표로 보기
df_display.style.background_gradient(
    subset=["precision", "recall", "mAP50", "mAP50-95"],
    cmap="RdYlGn", vmin=0, vmax=1
)

## 7. 성능 낮은 클래스 하이라이트

In [ ]:
weak = df[df["mAP50"].notna() & (df["mAP50"] < 0.3)].sort_values("mAP50")

if len(weak):
    print(f"⚠️  mAP50 < 0.3 인 클래스: {len(weak)}개\n")
    for _, row in weak.iterrows():
        print(f"  [{int(row['class_id']):>3}] {row['class_name']:30s}  "
              f"mAP50={row['mAP50']:.4f}  P={row['precision']:.4f}  R={row['recall']:.4f}")
else:
    print("✅ 모든 클래스 mAP50 >= 0.3")

## 8. 결과 파일로 저장
다른 컴퓨터 결과와 합쳐서 비교할 때 사용합니다.

In [ ]:
import json

# CSV (엑셀에서 보기 좋음)
csv_path = run_dir / "eval_results.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"📄 클래스별 CSV: {csv_path}")

# JSON (다른 결과와 통합용)
summary = {
    "model": MODEL,
    "overall": {
        "precision_mean": float(metrics.box.mp),
        "recall_mean":    float(metrics.box.mr),
        "mAP50":          float(metrics.box.map50),
        "mAP75":          float(metrics.box.map75),
        "mAP50-95":       float(metrics.box.map),
        "speed_ms_total": float(speed_total),
        "speed_detail":   {k: float(v) for k, v in metrics.speed.items()},
    },
    "per_class": df.to_dict(orient="records"),
}
json_path = run_dir / "eval_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"📄 요약 JSON: {json_path}")

print(f"\n🎉 완료!")
print(f"\n📂 결과 위치: {run_dir}")
print(f"   ├─ weights/best.pt          최적 모델")
print(f"   ├─ results.csv, results.png 학습 곡선")
print(f"   ├─ confusion_matrix.png     혼동 행렬")
print(f"   ├─ PR_curve.png             Precision-Recall 커브")
print(f"   ├─ eval_results.csv         클래스별 상세 지표")
print(f"   └─ eval_results.json        통합 비교용")

## 9. (선택) 학습 곡선 직접 그려보기

In [ ]:
import matplotlib.pyplot as plt

results_csv = run_dir / "results.csv"
if results_csv.exists():
    hist = pd.read_csv(results_csv)
    hist.columns = hist.columns.str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Loss
    loss_col = next((c for c in hist.columns if "train/box_loss" in c), None)
    if loss_col:
        axes[0].plot(hist["epoch"], hist[loss_col], label="train box loss")
    val_loss_col = next((c for c in hist.columns if "val/box_loss" in c), None)
    if val_loss_col:
        axes[0].plot(hist["epoch"], hist[val_loss_col], label="val box loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{MODEL} - Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

    # mAP
    for col_pattern, label in [("metrics/mAP50(", "mAP50"),
                                 ("metrics/mAP50-95(", "mAP50-95")]:
        col = next((c for c in hist.columns if col_pattern in c), None)
        if col:
            axes[1].plot(hist["epoch"], hist[col], label=label)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP")
    axes[1].set_title(f"{MODEL} - Validation mAP"); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print(f"results.csv 없음: {results_csv}")

In [2]:
!pip install ultralytics

  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 21.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.7 MB/s  0:00:01m0:00:0100:01
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics] [ultralytics]


In [4]:
import time
import torch
from pathlib import Path
from ultralytics import YOLO

# 1. 경로 설정 (각자 경로로 수정)
best_pt = Path("/data/runs/yolo11n-obb/weights/best.pt")
val_dir = Path("/data/yolo_obb_compressed/val/images")

# ────────────────────────────────────────────────────
# 2. 모델 로드 및 데이터 준비
model = YOLO(str(best_pt))
val_images = sorted(val_dir.glob("*.*"))

if not val_images:
    print("❌ 지정된 경로에 이미지가 없습니다.")
else:
    test_img = val_images[0]
    print(f"✅ 모델: {best_pt}")
    print(f"📷 이미지: {test_img.name}\n")

# 3. VRAM 상태 출력 함수
def print_vram(tag=""):
    if torch.cuda.is_available():
        alloc    = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved()  / 1024**2
        total    = torch.cuda.get_device_properties(0).total_memory / 1024**2
        print(f"[VRAM {tag}]  사용: {alloc:.1f} MB  /  예약: {reserved:.1f} MB  /  전체: {total:.1f} MB  /  여유: {total-reserved:.1f} MB")
    else:
        print("[VRAM] CPU 추론 중")

# 4. 성능 측정 시작
torch.cuda.empty_cache()
print_vram("추론 전")

# 워밍업 (GPU 초기화 및 연산 최적화)
for _ in range(3):
    _ = model(test_img, verbose=False)

# 추론 시간 측정
if torch.cuda.is_available():
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    times = []
    for _ in range(20):
        s.record()
        _ = model(test_img, verbose=False)
        e.record()
        torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
else:
    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        _ = model(test_img, verbose=False)
        times.append((time.perf_counter() - t0) * 1000)

# 5. 결과 출력
avg = sum(times) / len(times)
print(f"\n{'='*40}")
print(f"  평균 추론 시간 : {avg:.2f} ms")
print(f"  최소 / 최대    : {min(times):.2f} / {max(times):.2f} ms")
print(f"  평균 FPS       : {1000/avg:.1f} fps")
print(f"{'='*40}\n")
print_vram("추론 후")

✅ 모델: /data/runs/yolo11n-obb/weights/best.pt
📷 이미지: 101_10_0012fa07-d9bc-4b31-841b-5cd7a56b1158.jpg

[VRAM 추론 전]  사용: 10.3 MB  /  예약: 30.0 MB  /  전체: 7786.2 MB  /  여유: 7756.2 MB

  평균 추론 시간 : 24.04 ms
  최소 / 최대    : 19.78 / 29.10 ms
  평균 FPS       : 41.6 fps

[VRAM 추론 후]  사용: 52.7 MB  /  예약: 120.0 MB  /  전체: 7786.2 MB  /  여유: 7666.2 MB
